# **DLO-JZ Optimizers and Large Batch**

<div class="alert alert-block alert-success">

## Pratical notebook objectives

The purpose of this notebook is to compare optimizer solutions and parameters on small and large batches for a fixed number of iterations, as well as to learn to interpret the output of a learning-rate finder. This practical session is intended to be somewhat exploratory.

This pratical assignment will be conducted by modifying the **[dlojz1.py](dlojz1.py)** code. The cells in this notebook are *not* meant to be modified, except for rare exceptions indicated in the comments.

<div class="alert alert-block alert-info">

## Description

You are provided with a base [dlojz1.py](dlojz1.py) script - containing the training loop for a resnet18 model on the cifar10 dataset - as well as the [cifar10_Gugger_lrfinder.py](cifar10_Gugger_lrfinder.py) script containing an implementation of the *lr_finder* tool.

The objectives are threefold:
1. Learn which learning rate to choose based on the lr_finder curve.
2. Visualise the impact of the one-cycle lr-scheduler on training.
3. Visualize the difference in the training curve of different optimizers (SGD, AdamW).
4. See how weight decay and momentum affect the optimal learning rate value.

Each time, we will look at the case of **Small Batch** learning and **Large Batch** learning.

 * **Small Batch**: *Global Batch Size* of **256** on 1 GPU over **30** *epochs* (or **5880** *training steps*)
 * **Large Batch**: *Global Batch Size* of **8192** on 1 GPU over **75** *epochs* (or **525** *training steps*)

</div>

</div>

*Notebook written by the IDRIS AI assistance team. Created October 2022. Last revision autumn 2026*

The *python* functions for SLURM queue management developed by IDRIS and the functions dedicated to the DLO-JZ training are to be imported.

The environment module for the *jobs* and the image size are set for this *notebook*.
<div class="alert alert-block alert-warning">
    
**TODO:** choose a *nickname* (maximum 5 characters) to differentiate yourself in the SLURM queue and in collaborative tools during the training and competition.

</div>

In [ ]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import plot_accuracy, plot_accuracy_lr, model_vizu, update_vizu
from plot_lrfinder import plot_lrfinder
MODULE = 'pytorch-gpu/py3/2.8.0'
account = 'for@a100'
n_gpu = 1
name = 'pseudo'  #TODO

## Dataset and Model

For this lab, we will use the CIFAR 10 database and the Resnet-18 model to be able to run complete trainings in a reasonable amount of time.

### cifar10

In [ ]:
from dataset_and_model import train_dataset, train_loader, val_dataset, print_sample, resnet18
print("== Train set ==\n", train_dataset)
print("\n== Validation set ==\n", val_dataset)
print_sample(train_loader)

### Resnet-18

In [ ]:
print('number of total parameters: {}'.format(sum([p.numel() for p in resnet18.parameters()])))
print('number of trainable parameters: {}'.format(sum([p.numel() for p in resnet18.parameters() if p.requires_grad])))

<div class="alert alert-block alert-info">
    
## Reference training: Constant *Learning Rate* SGD
### Learning Rate Finder (SGD)
Whether with a constant *learning rate* or with a *Cycle Scheduler*, we first need to find the range of *learning rate* values that will have a positive effect on the model's learning.

We provided and implementation of a learning-rate finder based on Sylvain Gugger's post [How Do You Find a Good Learning Rate? (2018)](https://sgugger.github.io/how-do-you-find-a-good-learning-rate.html) in the [cifar10_Gugger_lrfinder.py](cifar10_Gugger_lrfinder.py) script.

After choosing a starting (min) and end (max) learning rates for the *lr_finder*, you will be able to plot the curves and identify a scale of *learning rates* that perform better than others.
For "similar" performances with the *lr finder*, between two *learning rates*, **the smaller one will probably have a more stable behavior**.

<div class="alert alert-block alert-warning">

**TODO**: Choose a minimum and maximum learning rate for the lr-finder. Weight decay and momentum are left at their default of 0 and 0.9 (resp.) for the time being.

</div>
</div>

In [ ]:
min_lr = ...
max_lr = ...

### Submitting the job
<u>**Please note that you will be utilising the compute nodes at this point**.</u>

To submit the job, please switch the following cell from `Raw NBConvert` mode to `Code` mode (shortcut: press the `Y` key whilst the cell is selected)

In [ ]:
command = [f'cifar10_Gugger_lrfinder.py -e 30 -b 256 -o SGD --min-lr {min_lr} --max-lr {max_lr}', 
           f'cifar10_Gugger_lrfinder.py -e 75 -b 8192 -o SGD --min-lr {min_lr} --max-lr {max_lr}']
jobids_ref = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                   account=account, time_max='00:10:00')
print(f'jobids_ref = {jobids_ref}')

In [ ]:
display_slurm_queue(name)

Then, switch the previous cell back to `Raw NBConvert` mode to avoid accidentally restarting a job (shortcut: press the `R` key whilst the cell is selected).

### Visualize the lr-finder's results

#### Small batch

In [ ]:
plot_lrfinder(jobids_ref[0])

#### Large batch

In [ ]:
plot_lrfinder(jobids_ref[1])

### Training (SGD)

We will launch a reference training with a constant *learning rate*.

<u>**Please note that you will be utilising the compute nodes at this point**.</u>

To submit the job, please switch the following cell from `Raw NBConvert` mode to `Code` mode (shortcut: press the `Y` key whilst the cell is selected)
<div class="alert alert-block alert-warning">

**TODO**: Choose a constant learning-rate based on the results from the *lr_finder*. 

</div>

In [ ]:
lr_smallb = ...
lr_largeb = ...
command = [f'dlojz1.py -b 256 -e 30 --wd 0. --lr {lr_smallb}', 
           f'dlojz1.py -b 8192 -e 75 --wd 0. --lr {lr_largeb}']
jobids_ref = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                   account=account, time_max='00:10:00')
print(f'jobids_ref = {jobids_ref}')

Then, switch the previous cell back to `Raw NBConvert` mode to avoid accidentally restarting a job (shortcut: press the `R` key whilst the cell is selected).

In [ ]:
display_slurm_queue(name)

### Plot accuracies

#### Small batch

In [ ]:
plot_accuracy_lr([jobids_ref[0]])

#### Large batch

In [ ]:
plot_accuracy_lr([jobids_ref[1]])

<div class="alert alert-block alert-info">

## Optimization n°1: *One Cycle Learning Rate*

We will now modify the code to replace the constant *learning rate* with a *One Cycle Scheduler* and compare the result with the reference training.

**We will then run trainings with and without weight decay** so you can estimate the effect of *weight decay* + *one cycle scheduler* on the *training* curve.</br>
Then, you can compare the *test accuracy* and *learning rate* curves with the reference training above.

<div class="alert alert-block alert-warning">

**TODO**: in the script [dlojz1.py](dlojz1.py): implement the OneCyle LR Scheduler by creating a `scheduler` object of the `torch.optim.lr_scheduler.OneCycleLR` class.

[Link to the Pytorch documentation](https://docs.pytorch.org/docs/2.14/generated/torch.optim.lr_scheduler.OneCycleLR.html)

/!\ Parameters that have a default value not set to **None** can be ignored for this practical.

Parameters can be obtained as follows:
* Epochs: `args.epochs`
* Batch size: `args.batch_size` or `mini_batch_size`
* Learning rate: `args.lr`
* momentum: `args.mom`
* Weight decay: `args.wd`
* Batch per epoch: `len(train_loader)` or `N_Batch`

(Click on the hints to reveal them)
<details>
    <summary><b>Hint 1: max_lr</b></summary>
    &nbsp;&nbsp;&nbsp;&nbsp;PyTorch's *OneCycleLR* automatically calculates a minimum *learning rate* from the given maximum value.</br>
    &nbsp;&nbsp;&nbsp;&nbsp;The max_lr is the learning rate chosen previously via the finder.
</details>
<details>
    <summary><b>Hint 2: lr update</b></summary>
    &nbsp;&nbsp;&nbsp;&nbsp;Don't forget to tell the scheduler when to update using scheduler.step().
</details>
<details>
    <summary><b>Hint 3: update timing</b></summary>
    &nbsp;&nbsp;&nbsp;&nbsp;Cyclic schedulers update the learning rate after each batch/step, not at the end of each epoch.
</details>
<details>
    <summary><b>Solution</b></summary>
    
&nbsp;&nbsp;&nbsp;&nbsp;After the dataloaders are created, create the scheduler with:
```python
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=args.lr, steps_per_epoch=N_batch, epochs=args.epochs)
```
</br>&nbsp;&nbsp;&nbsp;&nbsp;Then, before the validation step, add:
```python
scheduler.step()
```

</details>

</div>

<div class="alert alert-block alert-warning">

**TODO**: Choose learning-rates for small and large batches based on the results from the *lr_finder*. 

</div>
</div>

In [ ]:
lr_smallb = ...
lr_largeb = ...
command = [f'dlojz1_opti_1.py -b 256 -e 30 --wd 0. --lr {lr_smallb}', f'dlojz1_opti_1.py -b 256 -e 30 --wd 5e-4 --lr {lr_smallb}', 
           f'dlojz1_opti_1.py -b 8192 -e 75 --wd 0. --lr {lr_largeb}, 'f'dlojz1_opti_1.py -b 8192 -e 75 --wd 5e-4 --lr {lr_largeb}']
jobid_sgd = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                   account=account, time_max='00:10:00')
print(f'jobid_sgd = {jobid_sgd}')

In [ ]:
display_slurm_queue(name)

#### Small Batch

In [ ]:
plot_accuracy_lr([jobid_sgd[0],jobid_sgd[1]])

#### Large Batch

In [ ]:
plot_accuracy_lr([jobid_sgd[2],jobid_sgd[3]])

<div class="alert alert-block alert-info">

## Optimization n°2: *AdamW* Optimizer
We will now modify the optimizer to use *AdamW*.

<div class="alert alert-block alert-warning">
    
**TODO**: in the script [dlojz1.py](dlojz1.py):
Find the line declaring the *Stochastic Gradient Descent* optimizer and replace it with AdamW.

[Link to the Pytorch documentation](https://docs.pytorch.org/docs/2.8/generated/torch.optim.AdamW.html)

Parameters can be obtained as follows:
* Epochs: `args.epochs`
* Batch size: `args.batch_size` or `mini_batch_size`
* Learning rate: `args.lr`
* momentum: `args.mom`
* Weight decay: `args.wd`
* Batch per epoch: `len(train_loader)` or `N_Batch`

(Click on the hints to reveal them)
<details>
    <summary><b>Hint: momentum</b></summary>
&nbsp;&nbsp;&nbsp;&nbsp;The first value of parameter <b>betas</b> is the momentum. The second number can be left at its default value of <b>0.999</b>.
</details>

<details>
    <summary><b>Solution:</b></summary>
    
&nbsp;&nbsp;&nbsp;&nbsp;Replace
```python
optimizer = torch.optim.SGD(model.parameters(), args.lr, momentum=args.mom, weight_decay=args.wd)
```
</br>&nbsp;&nbsp;&nbsp;&nbsp;With the *AdamW* optimizer:
```python
optimizer = torch.optim.AdamW(model.parameters(), args.lr, betas=(args.mom, 0.999), weight_decay=args.wd)
```
</details>
</div>
</div>

### Learning Rate Finder (AdamW)
Since the optimizer has changed, we need to determine the maximum *learning rate* value to provide as a parameter. We will use the same `min_lr` and `max_lr` as for SGD.

### Submitting the job
<u>**Please note that you will be utilising the compute nodes at this point**.</u>

To submit the job, please switch the following cell from `Raw NBConvert` mode to `Code` mode (shortcut: press the `Y` key whilst the cell is selected)

In [ ]:
command = [f'cifar10_Gugger_lrfinder.py -e 30 -b 256 -o AdamW --min-lr {min_lr} --max-lr {max_lr}', 
           f'cifar10_Gugger_lrfinder.py -e 75 -b 8192 -o AdamW --min-lr {min_lr} --max-lr {max_lr}']
jobids_ref = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                   account=account, time_max='00:10:00')
print(f'jobids_ref = {jobids_ref}')

In [ ]:
display_slurm_queue(name)

Then, switch the previous cell back to `Raw NBConvert` mode to avoid accidentally restarting a job (shortcut: press the `R` key whilst the cell is selected).

### Visualize the lr-finder's results

#### Small batch

In [ ]:
plot_lrfinder(jobids_ref[0])

#### Large batch

In [ ]:
plot_lrfinder(jobids_ref[1])

### Training (AdamW)
You can now launch the training with the *AdamW* optimizer.

### Submitting the job
<u>**Please note that you will be utilising the compute nodes at this point**.</u>

To submit the job, please switch the following cell from `Raw NBConvert` mode to `Code` mode (shortcut: press the `Y` key whilst the cell is selected)
<div class="alert alert-block alert-warning">

**TODO**: Choose learning-rates for small and large batches based on the results from the *lr_finder*. 

</div>

In [ ]:
lr_smallb = ...
lr_largeb = ...
command = [f'dlojz1.py -b 256 -e 30 --wd 1e-2 --lr {lr_smallb}', 
           f'dlojz1.py -b 8192 -e 75 --wd 1e-1 --lr {lr_largeb}']
jobid_adamw = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                   account=account, time_max='00:10:00')
print(f'jobid_adamw = {jobid_adamw}')

Then, switch the previous cell back to `Raw NBConvert` mode to avoid accidentally restarting a job (shortcut: press the `R` key whilst the cell is selected).

In [ ]:
display_slurm_queue(name)

You can compare the *test accuracy* and *train accuracy* curves with the previous trainings with *SGD*.

#### Small Batch

In [ ]:
plot_accuracy([jobid_sgd[1], jobid_adamw[0]])

#### Large Batch

In [ ]:
plot_accuracy([jobid_sgd[3], jobid_adamw[1]])

<div class="alert alert-block alert-info">

## Going further: Weight decay and momentum
Weight decay and momentum are two important parameters of common optimizers; their value will affect training **and can also change the optimal learning rate value**.

<div class="alert alert-block alert-warning">
    
Based on this, we invite you to play with these two values and see how they affect the optimal learning rate predicted by the *lr_finder* and training efficiency for both optimizers.

</div>
</div>